In [ ]:
# efficientnet_cow.py
import torch
import torch.nn as nn
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import time

# paths
train_dir = '/path/to/cow/train'
val_dir   = '/path/to/cow/val'
num_classes = 5  # set according to dataset

# transforms
train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_ds = datasets.ImageFolder(train_dir, transform=train_tf)
val_ds   = datasets.ImageFolder(val_dir, transform=val_tf)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=4)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# model: EfficientNet-B0
model = models.efficientnet_b0(pretrained=True)
# replace classifier
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features, num_classes)
)
model = model.to(device)

# optionally freeze feature extractor for initial epochs
for param in model.features.parameters():
    param.requires_grad = True  # set False to freeze, True to fine-tune

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# training loop (no model saving)
num_epochs = 12
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    model.train()
    t0 = time.time()
    running_loss = 0.0
    all_preds, all_labels = [], []
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    epoch_loss = running_loss / len(train_ds)
    epoch_acc = accuracy_score(all_labels, all_preds)
    train_losses.append(epoch_loss); train_accs.append(epoch_acc)

    # validation
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds); all_labels.extend(labels.cpu().numpy())
    val_loss = running_loss / len(val_ds)
    val_acc = accuracy_score(all_labels, all_preds)
    val_losses.append(val_loss); val_accs.append(val_acc)
    scheduler.step()
    print(f"Epoch {epoch+1}/{num_epochs}  train_loss={epoch_loss:.4f} train_acc={epoch_acc:.4f}  val_loss={val_loss:.4f} val_acc={val_acc:.4f}  time={(time.time()-t0):.1f}s")

# plot loss and accuracy curves
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(train_losses, label='train_loss'); plt.plot(val_losses, label='val_loss'); plt.legend(); plt.title('Loss')
plt.subplot(1,2,2)
plt.plot(train_accs, label='train_acc'); plt.plot(val_accs, label='val_acc'); plt.legend(); plt.title('Accuracy')
plt.show()